In [1]:
from typing import List, Dict

In [2]:
import tensorflow as tf

# List available hardware
devices: List = tf.config.list_physical_devices()
print(f"Devices found: {devices}")

Devices found: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
gpu_devices: List = tf.config.list_physical_devices("GPU")

if gpu_devices:
    print(f"GPU devices found: {gpu_devices}")

    for each_gpu_device in gpu_devices:
        tf.config.experimental.set_memory_growth(each_gpu_device, True)

else:
    print("Running on Local CPU")

GPU devices found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Remarks: How to Fetch the Dataset

We will use `Garbage Classification Dataset` from [Kaggle](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification) for this project

### Dataset Description

Messr chchangcs curated a dataset of 2527 image datasets of garbages. The dataset contains 6 garbage categories, each having various count of images. These categories include:
* cardboard
* glass
* metal
* paper
* plastic
* trash

Other than images, the dataset also contains five TXT files, namely:
* one-indexed-files-notrash_train.txt
* one-indexed-files-notrash_val.txt
* one-indexed-files-notrash_test.txt
* one-indexed-files.txt
* zero-indexed-files.txt


### Authentication

Authentication is needed to download this dataset from Kaggle

First, you will need a Kaggle account. You can sign up [here](https://www.kaggle.com/account/login)

After login, you can download your Kaggle API token at [here](https://www.kaggle.com/settings) by clicking on the "Generate new Token" button under the "API" section.

You have several options to authenticate:

Option 1: kagglehub.login()

```Python
import kagglehub

kagglehub.login()
```
This will prompt you to enter your Kaggle API token.

Option 2: Environment Variable

```Bash
export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx # Coped from the settings UI
```

Option 3: API Token File

Store your Kaggle API token obtained from your [setting page](https://www.kaggle.com/settings) to a file located in `~/.kaggle/access_token`

Option 4: Google Colab Secret

If you are running this notebook from Google Cloud, then you can store your API token in a Colab secret named `KAGGLE_API_TOKEN`.

Instructions on adding secrets in Colab can be found [here](https://www.googlecloudcommunity.com/gc/Cloud-Hub/How-do-I-add-secrets-in-Google-Colab-Enterprise/m-p/784866)

Option 5: Legacy API credentials file

From your [Kaggle Account Setting Page](https://www.kaggle.com/settings), under the "Legacy API Credentials", click on the "Create Legeacy API Key" button to generate a `kaggle.json` file and store it at `~/.kaggle/kaggle.json`

In [4]:
# Authenticate Kaggle

import os
from pathlib import Path


import kagglehub

is_authenticated: bool = False

api_key_path: Path = Path("~/.kaggle/kaggle.json").expanduser()
is_api_key_exists: bool = True if api_key_path.exists() else False

api_token_path: Path = Path("~/.kaggle/access_token").expanduser()
is_api_token_exists: bool = True if api_token_path.exists() else False

## Check if running in Colab
def is_colab() -> bool:
    """
    Small utility function to check if this notebook is running on Colab
    :return: Boolean indication if this script is running on Colab
    :rtype bool
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False

is_colab_secret_exists: bool = False
if is_colab():
    from google.colab import userdata

    try:
        is_colab_secret_exists = True if userdata.get("KAGGLE_API_TOKEN") else False
    except userdata.SecretNotFoundError:
        is_colab_secret_exists = False

is_token_env_exists: bool = True if os.getenv("KAGGLE_API_TOKEN") else False

is_token_via_login: bool = False
if all([
    not is_api_key_exists
    , not is_api_token_exists
    , not is_colab_secret_exists
    , not is_token_env_exists
]):
    # if everything above fails, then login interactively
    # this will ask you to input the Kaggle API Token from a UI
    kagglehub.login()
    is_token_via_login = True


if any([
    is_api_key_exists
    , is_api_token_exists
    , is_token_env_exists
    , is_colab_secret_exists
    , is_token_via_login
]):
    is_authenticated = True
    print("Kagglehub has been authenticated")

else:
    raise Exception(" 🛑 Kagglehub fails to authenticate")

Kagglehub has been authenticated
